In [3]:
import spacy
import pandas as pd



In [4]:
!python -m spacy info


============================== Info about spaCy ==============================

spaCy version    3.8.3                         
Location         d:\Document\NLP_Project\NER\venv\Lib\site-packages\spacy
Platform         Windows-10-10.0.26200-SP0     
Python version   3.11.9                        
Pipelines                                      



In [5]:
!python -m spacy download en_core_web_lg

^C


In [7]:
# Importing libraries

import spacy
from spacy.tokens import DocBin
from tqdm import tqdm
import re

nlp = spacy.blank("en") # load a new spacy model
db = DocBin() # create a DocBin object

In [8]:
import json

# Load the training data (json file)

with open('D:/Document/NLP_Project/NER/data/train_annotations.json') as fp:
  train_data = json.load(fp)

In [10]:
train_data

{'classes': ['ORG_YEAR', 'SPORTS_NAME', 'WINNING_POSITION'],
 'annotations': [['Raleigh Flynn, your outstanding performance in the Equestrian Dressage during the 2013 competition has earned you a well-deserved 1st Position. Your sportsmanship and zeal for the game are appreciated. Congratulations!\r',
   {'entities': [[51, 70, 'SPORTS_NAME'],
     [82, 86, 'ORG_YEAR'],
     [130, 133, 'WINNING_POSITION']]}],
  ['Clifford Hall, your outstanding performance in the Boxing during the 2013 competition has earned you a well-deserved 2nd Position. Your sportsmanship and zeal for the game are appreciated. Congratulations!\r',
   {'entities': [[51, 57, 'SPORTS_NAME'],
     [69, 73, 'ORG_YEAR'],
     [117, 120, 'WINNING_POSITION']]}],
  ['In recognition of their exceptional performance in the Rowing, Clifton Hamilton has been awarded the 2nd Position in the 2009 competition. We extend our hearty congratulations and best wishes for all future endeavors!\r',
   {'entities': [[55, 61, 'SPORTS_NAME'

In [11]:
# Load the Validation data (json file)

with open('D:/Document/NLP_Project/NER/data/train_annotations.json') as fp:
  validation_data = json.load(fp)

In [12]:
validation_data

{'classes': ['ORG_YEAR', 'SPORTS_NAME', 'WINNING_POSITION'],
 'annotations': [['Raleigh Flynn, your outstanding performance in the Equestrian Dressage during the 2013 competition has earned you a well-deserved 1st Position. Your sportsmanship and zeal for the game are appreciated. Congratulations!\r',
   {'entities': [[51, 70, 'SPORTS_NAME'],
     [82, 86, 'ORG_YEAR'],
     [130, 133, 'WINNING_POSITION']]}],
  ['Clifford Hall, your outstanding performance in the Boxing during the 2013 competition has earned you a well-deserved 2nd Position. Your sportsmanship and zeal for the game are appreciated. Congratulations!\r',
   {'entities': [[51, 57, 'SPORTS_NAME'],
     [69, 73, 'ORG_YEAR'],
     [117, 120, 'WINNING_POSITION']]}],
  ['In recognition of their exceptional performance in the Rowing, Clifton Hamilton has been awarded the 2nd Position in the 2009 competition. We extend our hearty congratulations and best wishes for all future endeavors!\r',
   {'entities': [[55, 61, 'SPORTS_NAME'

In [15]:
train_data['annotations']

[['Raleigh Flynn, your outstanding performance in the Equestrian Dressage during the 2013 competition has earned you a well-deserved 1st Position. Your sportsmanship and zeal for the game are appreciated. Congratulations!\r',
  {'entities': [[51, 70, 'SPORTS_NAME'],
    [82, 86, 'ORG_YEAR'],
    [130, 133, 'WINNING_POSITION']]}],
 ['Clifford Hall, your outstanding performance in the Boxing during the 2013 competition has earned you a well-deserved 2nd Position. Your sportsmanship and zeal for the game are appreciated. Congratulations!\r',
  {'entities': [[51, 57, 'SPORTS_NAME'],
    [69, 73, 'ORG_YEAR'],
    [117, 120, 'WINNING_POSITION']]}],
 ['In recognition of their exceptional performance in the Rowing, Clifton Hamilton has been awarded the 2nd Position in the 2009 competition. We extend our hearty congratulations and best wishes for all future endeavors!\r',
  {'entities': [[55, 61, 'SPORTS_NAME'],
    [101, 104, 'WINNING_POSITION'],
    [121, 125, 'ORG_YEAR']]}],
 ['We take immen

In [16]:
for text, annot in tqdm(train_data['annotations']):
    doc =nlp.make_doc(text)
    ents = []
    for start,end, label in annot["entities"]:
        span = doc.char_span(start, end, label=label, alignment_mode="contract")
        if span is None:
            print("Skipping entity")
        else:
            ents.append(span)
            
    doc.ents = ents
    db.add(doc)
    
db.to_disk("./training_data.spacy") # save the docbin object

100%|██████████| 268/268 [00:00<00:00, 2946.20it/s]


In [17]:
# Converting test data into .spacy format

for text, annot in tqdm(validation_data['annotations']): 
    doc = nlp.make_doc(text) 
    ents = []
    for start, end, label in annot["entities"]:
        span = doc.char_span(start, end, label=label, alignment_mode="contract")
        if span is None:
            print("Skipping entity")
        else:
            ents.append(span)
    doc.ents = ents 
    db.add(doc)

db.to_disk("./validation_data.spacy") # save the docbin object

100%|██████████| 268/268 [00:00<00:00, 6612.86it/s]


In [23]:
# Getting the config file for training

!python -m spacy init config config.cfg --lang en --pipeline ner --optimize efficiency --force

⚠ To generate a more effective transformer-based config (GPU-only),
install the spacy-transformers package and re-run this command. The config
generated now does not use transformers.
ℹ Generated config template specific for your use case
- Language: en
- Pipeline: ner
- Optimize for: efficiency
- Hardware: CPU
- Transformer: None
✔ Auto-filled config with all values
✔ Saved config
config.cfg
You can now add your data and train your pipeline:
python -m spacy train config.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy


In [20]:
# Training
import os 

os.system("python -m spacy train config.cfg --output ./ --paths.train ./training_data.spacy --paths.dev ./validation_data.spacy")



1

In [24]:
!python -m spacy train config.cfg --output ./output --paths.train ./training_data.spacy --paths.dev ./validation_data.spacy

ℹ Saving to output directory: output
ℹ Using CPU

=========================== Initializing pipeline ===========================
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['tok2vec', 'ner']
ℹ Initial learn rate: 0.001
E    #       LOSS TOK2VEC  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  ------------  --------  ------  ------  ------  ------
  0       0          0.00     37.57    0.00    0.00    0.00    0.00
  1     200         29.28    858.77   99.50   99.50   99.50    1.00
  3     400         19.05     26.92   99.50   99.50   99.50    1.00
  6     600          7.05     14.87   99.81   99.88   99.75    1.00
 10     800         16.12     18.80   99.88  100.00   99.75    1.00
 14    1000         11.71     10.68   99.88   99.88   99.88    1.00
 20    1200         25.65     18.48   99.88  100.00   99.75    1.00
 27    1400         12.10     16.46  100.00  100.00  100.00    1.00
 35    1600        136.25     2

In [25]:
nlp = spacy.load("output/model-best")
print(nlp.pipe_names)
print(nlp.get_pipe("ner").labels)

['tok2vec', 'ner']
('ORG_YEAR', 'SPORTS_NAME', 'WINNING_POSITION')


In [26]:
test_texts = [
    "John Smith, your outstanding performance in the Boxing during the 2015 competition has earned you a well-deserved 1st Position. Congratulations!",
    "In recognition of their exceptional performance in the Rowing, Maria Lopez has been awarded the 3rd Position in the 2011 competition.",
]

for text in test_texts:
    doc = nlp(text)
    print(text)
    for ent in doc.ents:
        print(f"   {ent.text!r:25} -> {ent.label_}")
    print()

John Smith, your outstanding performance in the Boxing during the 2015 competition has earned you a well-deserved 1st Position. Congratulations!
   'Boxing'                  -> SPORTS_NAME
   '2015'                    -> ORG_YEAR
   '1st'                     -> WINNING_POSITION

In recognition of their exceptional performance in the Rowing, Maria Lopez has been awarded the 3rd Position in the 2011 competition.
   'Rowing'                  -> SPORTS_NAME
   '3rd'                     -> WINNING_POSITION
   '2011'                    -> ORG_YEAR



In [27]:
# Visual highlight
from spacy import displacy
displacy.render(nlp(test_texts[0]), style="ent", jupyter=True)

ImportError: cannot import name 'display' from 'IPython.core.display' (d:\Document\NLP_Project\NER\venv\Lib\site-packages\IPython\core\display.py)

In [28]:
# Reusable function: text -> dict of entities
def extract(text):
    doc = nlp(text)
    result = {}
    for ent in doc.ents:
        result.setdefault(ent.label_, []).append(ent.text)
    return result

extract("Clara Dean, your performance in the Archery during the 2018 competition has earned you a 2nd Position.")

{'SPORTS_NAME': ['Archery'], 'ORG_YEAR': ['2018'], 'WINNING_POSITION': ['2nd']}